# Step 1 Diagnostic — Can LSR Training Improve Retrieval?

Trains a **single retriever** on clean NFCorpus data — no federation, no noise, no Flower.

Sweeps 2 retrievers × 2 learning rates and measures MRR/NDCG before and after.

**Gate**: At least one variant must show >5% relative MRR improvement.

Runs `diagnostic_single_client.py` from the repo as a subprocess (same pattern as `single_alpha_kaggle_colab.ipynb`).

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/Abishek-Chakravarthy/fed-rag.git"
REPO_BRANCH = "q-fedrag2"

WORKSPACE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
REPO_DIR = WORKSPACE / "fed-rag"
EXP_DIR = REPO_DIR / "zz_coderuns" / "quality_aware_fedrag" / "mechanism_benchmark"
EXPORT_DIR = WORKSPACE / "diagnostic_exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"WORKSPACE: {WORKSPACE}")
print(f"REPO_DIR : {REPO_DIR}")
print(f"EXP_DIR  : {EXP_DIR}")

WORKSPACE: /kaggle/working
REPO_DIR : /kaggle/working/fed-rag
EXP_DIR  : /kaggle/working/fed-rag/zz_coderuns/quality_aware_fedrag/mechanism_benchmark


In [2]:
import os
import shutil
import subprocess
import sys

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        REPO_BRANCH,
        "--single-branch",
        REPO_URL,
        str(REPO_DIR),
    ],
    check=True,
)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "protobuf>=4.25.3,<6",
    "accelerate",
    "datasets<3.0.0",
    "flwr==1.22.0",
    "pyarrow",
    "pydantic",
    "pydantic-settings",
    "transformers==4.48.0",
    "sentence-transformers==3.4.1",
    "peft",
    "matplotlib",
    "pandas",
    "tqdm",
], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "--no-deps"], check=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
print("Clone + install complete")

Cloning into '/kaggle/working/fed-rag'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
a2a-sdk 0.3.25 requires protobuf>=5.29.5, but you have protobuf 4.25.9 which is incompatible.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2024.6.1 which is incompatible.
black 26.3.1 requires pathspec>=1.0.0, but you have pathspec 0.12.1 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 44.0.3 which is incompati

Clone + install complete


In [3]:
import importlib.metadata
import torch

print("torch version:", torch.__version__)
print("protobuf version:", importlib.metadata.version("protobuf"))
print("cuda available:", torch.cuda.is_available())
print("mps available :", torch.backends.mps.is_available())

if torch.cuda.is_available():
    print("gpu device:", torch.cuda.get_device_name(0))
    subprocess.run(["nvidia-smi"])
elif torch.backends.mps.is_available():
    print("Using Apple Metal backend")
else:
    print("WARNING: No GPU backend detected. Switch the notebook runtime to GPU.")

torch version: 2.10.0+cu128
protobuf version: 4.25.9
cuda available: True
mps available : False
gpu device: Tesla T4
Fri Apr 24 06:42:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                      

In [4]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-u",
    "diagnostic_single_client.py",
]

print("Running:", " ".join(cmd))
print(f"CWD: {EXP_DIR}")
print()

process = subprocess.Popen(
    cmd,
    cwd=EXP_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

process.wait()

if process.returncode != 0:
    raise subprocess.CalledProcessError(process.returncode, cmd)

print("\nDiagnostic complete!")

Running: /usr/bin/python3 -u diagnostic_single_client.py
CWD: /kaggle/working/fed-rag/zz_coderuns/quality_aware_fedrag/mechanism_benchmark


--- Launching variant 1a as subprocess ---
2026-04-24 06:42:48.589342: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777012968.831356      88 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777012968.902278      88 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777012969.442064      88 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777012969.442145      88 computation_placer.cc:177] computation p

In [5]:
import json
import shutil

summary_path = EXP_DIR / "diagnostic_results" / "diagnostic_summary.json"

if summary_path.exists():
    with open(summary_path) as f:
        results = json.load(f)

    print(f"{'ID':<4} {'Retriever':<42} {'LR':<8} {'Pre MRR':>9} {'Post MRR':>9} {'delta%':>7} {'Verdict'}")
    print("-" * 95)
    for r in results:
        short = r['retriever'].split('/')[-1]
        print(
            f"{r['id']:<4} {short:<42} {r['lr']:<8.0e} "
            f"{r['pre_test_mrr']:>9.6f} {r['post_test_mrr']:>9.6f} "
            f"{r['test_mrr_delta_pct']:>+6.1f}% {r['verdict']}"
        )

    # Copy to export dir
    shutil.copy2(summary_path, EXPORT_DIR / "diagnostic_summary.json")
    print(f"\nExported to: {EXPORT_DIR / 'diagnostic_summary.json'}")
else:
    print(f"Summary not found at {summary_path}")

ID   Retriever                                  LR         Pre MRR  Post MRR  delta% Verdict
-----------------------------------------------------------------------------------------------
1a   paraphrase-MiniLM-L3-v2                    2e-06     0.015989  0.014857   -7.1% DEGRADED
1b   paraphrase-MiniLM-L3-v2                    5e-07     0.015989  0.015756   -1.5% DEGRADED
1c   all-MiniLM-L6-v2                           2e-06     0.024253  0.026760  +10.3% IMPROVED
1d   all-MiniLM-L6-v2                           5e-07     0.024253  0.024242   -0.0% DEGRADED

Exported to: /kaggle/working/diagnostic_exports/diagnostic_summary.json


## What to download

Download `diagnostic_exports/diagnostic_summary.json` — it contains the full results for all 4 variants.

Use the best variant's retriever + LR for Step 2 (federated alpha sweep).